# 126 — Crítica, revisión y debate controlado

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** n=3: 0.6³ + 3·0.6²·0.4 = 0.216 + 0.432 = **0.648**.
n=5: Σ k=3..5 C(5,k)·0.6ᵏ·0.4⁵⁻ᵏ = 0.3456 + 0.2592 + 0.07776 ≈ **0.683**.
Ganancia: +3.5 pts por 2 llamadas ≈ **0.57 llamadas por punto**... es decir, ~1.75
puntos por llamada extra — y decrece con n.

**Ejercicio 2.** (a) empírico ≈ 0.784 (coincide con la teoría). (b) los tres aciertan
o fallan juntos: P(mayoría) = P(pregunta fácil) = **0.70** — exactamente p individual.
La correlación total anula el beneficio del voto: es el caso límite del error
sistemático compartido.

**Ejercicio 3.** Rúbrica de ejemplo: (1) el finding cita evidencia verificable
[bloqueante]; (2) severidad justificada respecto al score [mayor]; (3) accionable: se
deduce qué corregir [mayor]; (4) sin afirmaciones fuera del alcance revisado [menor].
"falta threat model": cumple 3 y 4; falla 1 (no cita dónde buscó) y 2 (¿por qué 0.6 y
no 0.3?) → **revisar** con dos hallazgos.

**Ejercicio 4.** Mayoritaria "12" con registro `{"12": 2, "15": 1}` — el disenso se
conserva como señal de incertidumbre. En empate 1-1-1 no hay señal mayoritaria:
elegir al azar fabrica una confianza que no existe; escalar (otra ronda, un juez con
rúbrica, o un humano) convierte la incertidumbre en decisión informada.


In [ ]:
result = run_lab("multiagent", seed=126)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
from math import comb, ceil
import random

# Ejercicio 1
def p_mayoria(p, n):
    return sum(comb(n, k) * p**k * (1-p)**(n-k) for k in range(ceil(n/2), n+1))

print(f"n=3: {p_mayoria(0.6, 3):.3f}  n=5: {p_mayoria(0.6, 5):.3f}")

# Ejercicio 2
def simula(correlacionado, trials=10_000, seed=126):
    rng = random.Random(seed)
    aciertos = 0
    for _ in range(trials):
        if correlacionado:
            votos = [rng.random() >= 0.3] * 3          # fallan/aciertan juntos
        else:
            votos = [rng.random() < 0.7 for _ in range(3)]
        aciertos += sum(votos) >= 2
    return aciertos / trials

print(f"independientes: {simula(False):.3f} (teoría 0.784) | "
      f"correlacionados: {simula(True):.3f} (≈ p = 0.7)")

# Ejercicio 4
def debate(respuestas):
    conteo = {r: respuestas.count(r) for r in set(respuestas)}
    ganadora = max(conteo, key=conteo.get)
    empatadas = [r for r, v in conteo.items() if v == conteo[ganadora]]
    if len(empatadas) > 1:
        return {"decision": None, "motivo": "empate: escalar", "votos": conteo}
    return {"decision": ganadora, "votos": conteo}

print(debate(["12", "12", "15"]))
print(debate(["12", "15", "9"]))

# Conexión con el laboratorio: los workers evalúan aspectos distintos (no votan)
result = run_lab("multiagent", seed=126)
assert result["kind"] == "multiagent"
print("consolidación (no debate):", result["result"]["supervisor"])


## Reflexión

1. Los 3 workers del laboratorio *evalúan aspectos distintos* y no votan. ¿Qué cambiaría en el diseño para convertirlos en un debate al estilo Du et al., y qué pregunta del laboratorio admitiría voto mayoritario?
2. Si duplicas los votantes de 3 a 6 usando el mismo modelo y prompt, ¿por qué la mejora observada será menor que la que predice Condorcet? ¿Cómo diversificarías?
3. ¿En qué tareas esperarías que un solo agente con herramienta (calculadora, retrieval) supere a un debate de 5 agentes sin herramientas, y por qué?
